In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "munar2015common")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "munar_experiment3_full.csv")
complete_path_2 = os.path.join(original_data_pathway, "munar_experiment4_full.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

experiment_import = [[df1, '3', '84ms'],
                    [df2, '4','until_response']]
for x,y,k in experiment_import:
    x['experiment'] = y
    x['experiment_name'] = k



In [3]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x=x.rename(columns={"name": "ape",
        "sessiondate": "date",
        "blockorder": "block_order",
        "sessiontime": "session_time",
        "fixationrt": "fixation_rt",
        "imageleft": "image_left",
        "imageright": "image_right",
        "maskrtinclbuffer": "mask_rt_incl_buffer",
        "selectedpicture": "selected_picture",
        "stimuli.offsetdelay": "stimuli_offset_delay",
        "stimulibuffer.onsetdelay": "stimuli_buffer_onset_delay"})
    x['data_subset']="data_subset_" + str(index+1)
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [4]:
fulldf[['year','month', 'day']] = fulldf['date'].str.split('-',expand=True)

In [5]:
code_list=["block_order"]
for index, x in enumerate(code_list):    
    fulldf[x] = fulldf[x].astype(str)
    temp=[]
    for entry in fulldf[x]:
        if entry == '1' or entry=='1.0':
            entry = "Block_1-Block_2"
        elif entry =='2' or entry=='2.0':
            entry = "Block_2-Block_1"
        temp.append(entry)
    fulldf = fulldf.assign(temp_col=temp)
    fulldf=fulldf.rename(columns={'temp_col': x+'_codes'})

In [6]:
fulldf.replace('nan', np.nan, inplace=True)

In [7]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

In [8]:
fulldf.dropna(subset=['ape'], inplace=True)
# fulldf.columns
fulldf.rename(columns={"ape": "participant"}, inplace=True)

In [9]:
fulldf=fulldf.sort_values(by = ['session', 'order'])
fulldf.rename(columns={"trial": "trial_id",
                       "order":"trial"}, inplace=True)

comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

In [10]:
fulldf=fulldf[['study_id','experiment', 'experiment_name', 'year', 'month', 'day',
       'subject', 'participant', 'age_in_years','sex', 'species', 'session',  'trial', 'trial_id',
       'session_time', 'block_order','block_order_codes', 'block',  'fixation_rt', 'image_left',
       'image_right', 'mask_rt_incl_buffer', 'reward', 
       'selected_picture']]


In [11]:
for index in range(3,5):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'munar2015common_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'munar2015common_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)